In [ ]:
!pip install -q \
transformers==4.41.2 \
sentence-transformers==2.7.0 \
accelerate==0.30.1 \
faiss-cpu \
pypdf \
langchain \
langchain-text-splitters \
sentencepiece

In [ ]:
import os
import re
import warnings
warnings.filterwarnings("ignore")

# PDF reading
from pypdf import PdfReader

# Text splitting (UPDATED)
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Multilingual embeddings
from sentence_transformers import SentenceTransformer

# Vector store
import faiss
import numpy as np

# Language model
from transformers import pipeline

# Colab upload
from google.colab import files

print("✅ All imports successful!")

✅ All imports successful!


In [ ]:
import os

# 📂 Automatically detect all PDF files in Colab /content folder
pdf_files = [f for f in os.listdir('/content') if f.endswith('.pdf')]

if len(pdf_files) == 0:
    print("❌ No PDF files found in /content")
else:
    print(f"✅ Found {len(pdf_files)} PDF file(s):")

    for i, file in enumerate(pdf_files, 1):
        print(f"{i}. {file}")

# Select first PDF automatically
PDF_PATH = f"/content/sciadv.adv8031.pdf"

print(f"\n📄 Using PDF: {PDF_PATH}")

✅ Found 1 PDF file(s):
1. sciadv.adv8031.pdf

📄 Using PDF: /content/sciadv.adv8031.pdf


In [ ]:
def extract_text_from_pdfs(file_list):
    """
    Reads each PDF page by page and concatenates all text.
    Returns a single combined string from all uploaded PDFs.
    """
    all_text = ""
    for filename in file_list:
        print(f"📄 Extracting text from: {filename}")
        reader = PdfReader(filename)
        for page_num, page in enumerate(reader.pages):
            page_text = page.extract_text()
            if page_text:
                all_text += page_text + "\n"
        print(f"   ✅ Extracted {len(reader.pages)} pages from {filename}")
    return all_text

# Run extraction
raw_text = extract_text_from_pdfs(pdf_files)

# Quick sanity check
print(f"\n📊 Total characters extracted: {len(raw_text):,}")
print(f"📝 Preview (first 300 chars):\n{raw_text[:300]}...")

📄 Extracting text from: sciadv.adv8031.pdf
   ✅ Extracted 18 pages from sciadv.adv8031.pdf

📊 Total characters extracted: 126,712
📝 Preview (first 300 chars):
Stewart et al., Sci. Adv. 11, eadv8031 (2025)     25 June 2025
Science Adv AnceS  |  Review
1 of 17
ECOLOGY
The future of baleen whales: Recoveries, environmental 
constraints, and climate change
Joshua D. Stewart1*, M. Tim Tinker2,3, Robert L. Brownell Jr.4, Andrew J. Read5
Most baleen whales were ...


In [ ]:
import nltk
nltk.download('punkt_tab')

from nltk.tokenize import sent_tokenize

# Clean text first
def clean_text(text):
    text = text.replace("\n", " ")
    text = text.replace("- ", "")
    return text

raw_text = clean_text(raw_text)

# Sentence-based chunking
sentences = sent_tokenize(raw_text)

chunks = []
current_chunk = ""

for sentence in sentences:
    if len(current_chunk) + len(sentence) < 800:
        current_chunk += " " + sentence
    else:
        chunks.append(current_chunk.strip())
        current_chunk = sentence

if current_chunk:
    chunks.append(current_chunk.strip())

print(f"✅ Created {len(chunks)} clean sentence-based chunks")
print(f"📌 Example chunk:\n\n{chunks[0][:300]}...")

✅ Created 181 clean sentence-based chunks
📌 Example chunk:

Stewart et al., Sci. Adv. 11, eadv8031 (2025)     25 June 2025 Science Adv AnceS  |  Review 1 of 17 ECOLOGY The future of baleen whales: Recoveries, environmental  constraints, and climate change Joshua D. Stewart1*, M. Tim Tinker2,3, Robert L. Brownell Jr.4, Andrew J. Read5 Most baleen whales were ...


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
print("⏳ Loading multilingual embedding model (this may take a minute)...")

# Load the multilingual sentence transformer model
embedding_model = SentenceTransformer("intfloat/multilingual-e5-base")

print("✅ Embedding model loaded!")

def generate_embeddings(text_list, prefix="passage"):
    """
    Generates embeddings for a list of texts.

    For multilingual-e5, prepend:
      - 'passage: ' for document chunks
      - 'query: '   for user queries
    This improves retrieval quality significantly.

    Args:
        text_list : List of strings to embed
        prefix    : 'passage' for documents, 'query' for queries

    Returns:
        numpy array of shape (len(text_list), embedding_dim)
    """
    # Prepend the required prefix for e5 models
    prefixed = [f"{prefix}: {t}" for t in text_list]
    embeddings = embedding_model.encode(
        prefixed,
        show_progress_bar=True,
        normalize_embeddings=True   # Normalize for cosine similarity via inner product
    )
    return np.array(embeddings, dtype="float32")

# Embed all document chunks
print("\n⏳ Generating embeddings for document chunks...")
chunk_embeddings = generate_embeddings(chunks, prefix="passage")

print(f"✅ Generated embeddings: shape = {chunk_embeddings.shape}")

⏳ Loading multilingual embedding model (this may take a minute)...
✅ Embedding model loaded!

⏳ Generating embeddings for document chunks...


Batches:   0%|          | 0/6 [00:00<?, ?it/s]

✅ Generated embeddings: shape = (181, 768)


In [ ]:
def build_faiss_index(embeddings):
    """
    Builds a FAISS index using Inner Product (equivalent to cosine similarity
    when embeddings are L2-normalized).

    Args:
        embeddings: numpy float32 array of shape (N, D)

    Returns:
        faiss.IndexFlatIP index
    """
    embedding_dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(embedding_dim)   # IP = Inner Product
    index.add(embeddings)
    return index

# Build the index
faiss_index = build_faiss_index(chunk_embeddings)

print(f"✅ FAISS index built!")
print(f"   Total vectors indexed: {faiss_index.ntotal}")

✅ FAISS index built!
   Total vectors indexed: 181


In [ ]:
def retrieve_relevant_chunks(query, index, chunks, top_k=8):
    """
    Retrieves the top-k most relevant document chunks for a given query.

    Steps:
      1. Embed the query using the same model (with 'query:' prefix)
      2. Search the FAISS index for nearest neighbors
      3. Return the matching text chunks

    Args:
        query   : User's question (any language)
        index   : FAISS index containing chunk embeddings
        chunks  : List of original text chunk strings
        top_k   : Number of top results to retrieve

    Returns:
        List of (score, chunk_text) tuples
    """
    # Embed the query — use 'query' prefix for e5 models
    query_embedding = generate_embeddings([query], prefix="query")

    # Search the FAISS index
    scores, indices = index.search(query_embedding, top_k)

    # Collect results
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx != -1:  # -1 means no result found
            results.append((float(score), chunks[idx]))

    return results

# Quick test
print("🔍 Testing retrieval with a sample English query...")
test_results = retrieve_relevant_chunks(
    "What is the main contribution of this paper?",
    faiss_index, chunks, top_k=3
)
for i, (score, chunk) in enumerate(test_results):
    print(f"\n--- Result {i+1} (score: {score:.4f}) ---")
    print(chunk[:200] + "...")

🔍 Testing retrieval with a sample English query...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


--- Result 1 (score: 0.8037) ---
DOI: 10.1126/sciadv.adv8031 View the article online https://www.science.org/doi/10.1126/sciadv.adv8031 Permissions https://www.science.org/help/reprints-and-permissions Use of this article is subject ...

--- Result 2 (score: 0.7910) ---
A. Goldbogen,   n. Gales, J. w. d urban, Sympatry and resource partitioning between the largest   krill consumers around the Antarctic Peninsula. Mar. Ecol. Prog. Ser. 669, 1–16 (2021). 25. P . n. Tra...

--- Result 3 (score: 0.7908) ---
Elaborations on this basic model include allowance for nonlinear  relationships between percapita growth rate and abundance relative to K [i.e., the theta logistic equation (31)], the incorporation of...


In [ ]:
!pip install transformers==4.41.2 sentencepiece --quiet

In [ ]:
#import os
#os.kill(os.getpid(), 9)

In [ ]:
from transformers import pipeline
print("⏳ Loading FLAN-T5 answer generation model...")

# Load FLAN-T5 as a text-to-text pipeline
qa_pipeline = pipeline(
    "text2text-generation",   # now this WILL work
    model="google/flan-t5-small",
    max_new_tokens=300,
    do_sample=False,
    temperature=0.2
)
print("✅ FLAN-T5 model loaded and ready!")

⏳ Loading FLAN-T5 answer generation model...
✅ FLAN-T5 model loaded and ready!


In [ ]:

def detect_language(text):
    """
    Detects the language of the input text using Unicode ranges.

    Unicode ranges:
      - Hindi (Devanagari): U+0900 to U+097F
      - Kannada:            U+0C80 to U+0CFF
      - Default:            English

    Args:
        text: Input string

    Returns:
        'hindi', 'kannada', or 'english'
    """
    devanagari_chars = len(re.findall(r'[\u0900-\u097F]', text))
    kannada_chars    = len(re.findall(r'[\u0C80-\u0CFF]', text))

    if kannada_chars > devanagari_chars and kannada_chars > 2:
        return "kannada"
    elif devanagari_chars > 2:
        return "hindi"
    else:
        return "english"

# Test language detection
print("🌐 Language Detection Tests:")
print(f"  'What is machine learning?' → {detect_language('What is machine learning?')}")
print(f"  'मशीन लर्निंग क्या है?' → {detect_language('मशीन लर्निंग क्या है?')}")
print(f"  'ಯಂತ್ರ ಕಲಿಕೆ ಎಂದರೇನು?' → {detect_language('ಯಂತ್ರ ಕಲಿಕೆ ಎಂದರೇನು?')}")

🌐 Language Detection Tests:
  'What is machine learning?' → english
  'मशीन लर्निंग क्या है?' → hindi
  'ಯಂತ್ರ ಕಲಿಕೆ ಎಂದರೇನು?' → kannada


In [ ]:
def build_prompt(query, context_chunks, language):

    context = " ".join([chunk.replace("\n", " ") for _, chunk in context_chunks[:3]])

    if language == "hindi":
        instruction = "नीचे दिए गए संदर्भ के आधार पर संक्षेप में हिंदी में उत्तर दें।"
    elif language == "kannada":
        instruction = "ಕೆಳಗಿನ ಮಾಹಿತಿಯ ಆಧಾರದಲ್ಲಿ ಸಂಕ್ಷಿಪ್ತವಾಗಿ ಕನ್ನಡದಲ್ಲಿ ಉತ್ತರಿಸಿ."
    else:
        instruction = "Answer briefly and clearly in English."

    prompt = f"""
Answer the question using ONLY the given context.

{instruction}

Context:
{context}

Question: {query}

Answer:
"""
    return prompt

In [ ]:
def generate_answer(query, faiss_index, chunks, top_k=8):

    # Step 1: Detect language
    language = detect_language(query)
    print(f"🌐 Detected language: {language.upper()}")

    # Step 2: Retrieve chunks
    print("🔍 Retrieving relevant context...")
    relevant_chunks = retrieve_relevant_chunks(query, faiss_index, chunks, top_k=top_k)

    # Step 3: Clean + limit context (IMPORTANT)
    context = " ".join([chunk.replace("\n", " ") for _, chunk in relevant_chunks[:3]])

    # Step 4: Build prompt
    if language == "hindi":
        instruction = "नीचे दिए गए संदर्भ के आधार पर संक्षेप में हिंदी में उत्तर दें।"
    elif language == "kannada":
        instruction = "ಕೆಳಗಿನ ಮಾಹಿತಿಯ ಆಧಾರದಲ್ಲಿ ಸಂಕ್ಷಿಪ್ತವಾಗಿ ಕನ್ನಡದಲ್ಲಿ ಉತ್ತರಿಸಿ."
    else:
        instruction = "Answer briefly and clearly in English."

    prompt = f"""
Answer the question using ONLY the given context.

{instruction}

Context:
{context}

Question: {query}

Answer:
"""

    # Step 5: Generate answer
    print("🤖 Generating answer...")
    output = qa_pipeline(prompt)
    answer = output[0]["generated_text"].strip()

    # 🔥 Step 6: Fix empty / wrong language output
    if language == "hindi" and (answer == "" or not any('\u0900' <= c <= '\u097F' for c in answer)):
        from transformers import pipeline
        translator = pipeline("translation", model="Helsinki-NLP/opus-mt-en-hi")
        answer = translator(context[:200])[0]['translation_text']

    return {
        "language": language,
        "answer": answer,
        "retrieved_chunks": relevant_chunks
    }

In [ ]:
def ask_question(query, show_context=False):
    """
    Main entry point for the QA system.

    Args:
        query       : Your question in English, Hindi, or Kannada
        show_context: If True, also prints the retrieved source chunks
    """
    print("\n" + "="*60)
    print(f"❓ Question: {query}")
    print("="*60)

    result = generate_answer(query, faiss_index, chunks)

    print(f"\n💬 Answer ({result['language'].upper()}):")
    print("-"*40)
    print(result["answer"])

    if show_context:
        print("\n📚 Retrieved Context Chunks:")
        print("-"*40)
        for i, (score, chunk) in enumerate(result["retrieved_chunks"], 1):
            print(f"\n[Chunk {i}] Relevance Score: {score:.4f}")
            print(chunk[:300] + ("..." if len(chunk) > 300 else ""))

    print("="*60 + "\n")
    return result

In [ ]:
result_en = ask_question(
    "How does climate change affect baleen whales?",
    show_context=True
)


❓ Question: How does climate change affect baleen whales?
🌐 Detected language: ENGLISH
🔍 Retrieving relevant context...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

🤖 Generating answer...

💬 Answer (ENGLISH):
----------------------------------------
densityindependent

📚 Retrieved Context Chunks:
----------------------------------------

[Chunk 1] Relevance Score: 0.8965
In turn, this could expose baleen whales to increased risk of acute human impacts such as vessel strikes and  fisheries interactions in regions where they were previously seen  rarely, as has been documented in North Atlantic right whales  (Eubalaena glacialis) (106). This is an example of a density...

[Chunk 2] Relevance Score: 0.8951
CLIMATE AND THE FUTURE OF BALEEN WHALE POPULATIONS In an unfortunate twist of fate, as humans largely ended the hunting  of baleen whales, we accelerated our emissions of greenhouse gases,  driving global warming and the rapid alteration of highlatitude  oceans that are primary foraging areas for ba...

[Chunk 3] Relevance Score: 0.8938
Recent studies have shown that baleen whale behavior is sensitive to changes in the distribution and seasonal ava

In [ ]:
result_en = ask_question(
    "मानव गतिविधियाँ बेलीन व्हेल्स को कैसे प्रभावित करती हैं?",
    show_context=True
)


❓ Question: मानव गतिविधियाँ बेलीन व्हेल्स को कैसे प्रभावित करती हैं?
🌐 Detected language: HINDI
🔍 Retrieving relevant context...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

🤖 Generating answer...

💬 Answer (HINDI):
----------------------------------------
सुधार प्रबंधन तेज़ी से बदलनेवाले वातावरण में पास होने के बारे में बेहतर समझ की आवश्‍यकता होगी कि मुख्य संसाधनों में परिवर्तन विशिष्ट अत्यावश्‍यक दरों में योग कैसे दिया जाता है, एक

📚 Retrieved Context Chunks:
----------------------------------------

[Chunk 1] Relevance Score: 0.7957
Improving management approaches in a rapidly  changing environment will require a better understanding of how  variation in key limiting resources translates into variation in specific vital rates, and thus population dynamics. With this in mind,  we ask: how are limiting resources—specifically prey...

[Chunk 2] Relevance Score: 0.7951
In turn, this could expose baleen whales to increased risk of acute human impacts such as vessel strikes and  fisheries interactions in regions where they were previously seen  rarely, as has been documented in North Atlantic right whales  (Eubalaena glacialis) (106). This is an example of a 

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr

# Wrapper function for Gradio
def gradio_answer(question):
    result = generate_answer(question, faiss_index, chunks)
    return result["answer"]

# Gradio UI
demo = gr.Interface(
    fn=gradio_answer,

    inputs=gr.Textbox(
        label="❓ Ask Question",
        placeholder="Ask in English, Hindi, or Kannada..."
    ),

    outputs=gr.Textbox(
        label="📝 Answer",
        lines=10
    ),

    title="🌐 Multilingual Research Paper QA System",

    description="Ask questions about the uploaded research paper."
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a18b35afeeeb9cdbf7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
processed = False

def process_pdf(file):
    global chunks, faiss_index, processed

    from pypdf import PdfReader
    reader = PdfReader(file.name)

    text = ""
    for page in reader.pages:
        text += page.extract_text()

    # Clean text
    text = text.replace("\n", " ").replace("- ", "")

    # Sentence chunking
    import nltk
    from nltk.tokenize import sent_tokenize
    nltk.download('punkt', quiet=True)

    sentences = sent_tokenize(text)

    chunks = []
    current_chunk = ""

    for sentence in sentences:
        if len(current_chunk) + len(sentence) < 800:
            current_chunk += " " + sentence
        else:
            chunks.append(current_chunk.strip())
            current_chunk = sentence

    if current_chunk:
        chunks.append(current_chunk.strip())

    # Embeddings (IMPORTANT CHANGE)
    texts = [f"passage: {c}" for c in chunks]
    embeddings = embedding_model.encode(texts, normalize_embeddings=True)

    import faiss
    import numpy as np

    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(np.array(embeddings).astype("float32"))

    faiss_index = index
    processed = True

    return "✅ PDF processed successfully!"